# Pipeline progress

Loads the snapshot written by

```bash
python -m bb_hpc.progress_report
```

and lets you poke at it. **Nothing is recomputed here** — every number in this notebook
comes from one atomic snapshot with a single `generated_at`, so cells can never disagree
with each other the way the old `bbb_and_tracking_progress.ipynb` did.

A unit reported *pending* here is exactly a unit that `<stage>_submit` would schedule:
both use the same completion predicates from `bb_hpc.src.generate`.

To recompute from inside the notebook, see the last section.

In [ ]:
from bb_hpc.src.progress import load_latest, load_history, build_report, resolve_paths

rep = load_latest()
print(f"snapshot generated {rep.generated_at:%Y-%m-%d %H:%M} UTC")
print(f"dates {rep.dates[0]} .. {rep.dates[-1]}  ({len(rep.dates)} days)")
print(f"resultdir {rep.config['resultdir']}")

## Overview

The same one-screen report the CLI prints.

In [ ]:
print(rep.render())

In [ ]:
rep.summary()

## Per-day tables

`stage.by_day` has `day, total, done, pending, skipped, pct`.

`pct` excludes *skipped* units — a camera with no frames can never produce a background,
so it must not drag the percentage down forever. It is counted in `skipped` instead.

The aliases below match the names the old notebook used.

In [ ]:
# familiar names from the old notebook
by_day        = rep.stages["detect"].by_day          # raw videos -> .bbb
sd_by_day     = rep.stages["save_detect"].by_day     # (cam, hour) -> detections parquet
trk_by_day    = rep.stages["tracking"].by_day        # (cam, hour) -> tracking dill
rpi_by_day    = rep.stages["rpi"].by_day

# new: comb / cell-seg stages
fe_by_day     = rep.stages["frame_extract"].by_day   # (date, cam) -> extracted frames
bg_by_day     = rep.stages["background"].by_day      # (date, cam) -> comb backgrounds

by_day.tail(10)

In [ ]:
# days that still need work, any stage
{name: sp.pending_dates for name, sp in rep.stages.items() if sp.available and sp.pending_dates}

## Drill into what is missing

`stage.units` is one row per work unit with a `status` column.

| stage | unit | statuses |
|---|---|---|
| `detect` | one raw video | `done` / `zero_byte_stub` / `missing` |
| `save_detect`, `tracking` | `(cam_id, hour window)` | `done` / `stale` / `missing` |
| `frame_extract` | `(date, cam)` | `done` / `pending` / `skipped_no_txt` |
| `background` | `(date, cam)` | `done` / `pending` / `skipped_no_frames` / `skipped_min_frames` |
| `rpi` | one RPi video | `done` / `missing` |

`stale` means the output is older than a `.bbb` in its window, so it will be redone.

In [ ]:
rep.stages["tracking"].pending_units().head(20)

In [ ]:
# which detections are 0-byte stubs (aborted writes -- these read as
# "done" in the old notebook, but detect_submit will not redo them without --check-read-bbb)
d = rep.stages["detect"].units
d[d["status"] == "zero_byte_stub"].head(20)

In [ ]:
rep.stages["background"].units.head(20)

In [ ]:
# warnings worth reading (e.g. a config_tag that could not be resolved canonically)
for name, sp in rep.stages.items():
    for note in sp.notes:
        print(f"[{name}] {note}")

## Resubmit what did not finish

These are printed by the CLI and written to `<snapshot>/commands.sh`. Copy-paste them,
or run one from here.

In [ ]:
for stage, cmd in rep.commands().items():
    print(f"# {stage}\n{cmd}\n")

In [ ]:
# a different backend, without recomputing
rep.commands(backend="slurm")

In [ ]:
# Run one. Uncomment deliberately -- this submits jobs.
# import subprocess, shlex
# cmd = rep.commands()["tracking"]
# subprocess.run(shlex.split(cmd), check=True)

## Progress over time

Every `progress_report` run appends one row to `progress/history.jsonl`.

In [ ]:
hist = load_history()
hist.tail(10)

In [ ]:
import matplotlib.pyplot as plt

pct_cols = [c for c in hist.columns if c.endswith("_pct")]
if len(hist) > 1 and pct_cols:
    ax = hist.plot(x="generated_at", y=pct_cols, marker="o", figsize=(10, 4))
    ax.set_ylabel("% complete")
    ax.set_ylim(0, 101)
    ax.legend([c.replace("_pct", "") for c in pct_cols], loc="lower right", ncol=3)
    plt.tight_layout()
else:
    print("Need at least two progress_report runs to plot a trend.")

## Recompute here

Normally you refresh from the shell (`python -m bb_hpc.progress_report --refresh videos`),
which also saves the snapshot. But you can rebuild in-process:

In [ ]:
from bb_hpc import settings

# rep = build_report(
#     **resolve_paths(),
#     dates=None,                       # None = every date in the catalogs
#     backend="k8s",
#     frame_extract_settings=settings.frame_extract_settings,
#     background_settings=settings.background_settings,
# )
# rep.save()      # overwrite the snapshot + append to history.jsonl